In [1]:
import pandas as pd

# Leggo il foglio "Table 1" del file Excel dell'ONU
# skiprows=10 salta le prime 10 righe (logo e titolo)
df = pd.read_excel(
    "dataraw/undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx",
    sheet_name="Table 1",
    skiprows=10
)

print("Righe e colonne:", df.shape)
df.head()

Righe e colonne: (28030, 31)


,Index,"Region, development group, country or area of destination",Coverage,Data type,Location code of destination,"Region, development group, country or area of origin",Location code of origin,1990,1995,2000,...,2020.1,2024.1,1990.2,1995.2,2000.2,2005.2,2010.2,2015.2,2020.2,2024.2
0,1,World,NaN,NaN,900,World,900,153916063.0,163176002,174566152,...,143223497,158009795,76143981,80632704,86275210,94338056,107763606,119854396,132060535,146012018
1,2,World,NaN,NaN,900,Sub-Saharan Africa,1834,14124662.0,15183742,14584913,...,14375255,16046945,6665704,7163563,6846928,7325563,8503651,10696751,12759702,14614665
2,3,World,NaN,NaN,900,Northern Africa and Western Asia,1833,14986109.0,17216219,18728264,...,21212741,22913639,6648687,7547605,8080447,9029322,10800712,13724647,15984112,17615687
3,4,World,NaN,NaN,900,Central and Southern Asia,1831,30342957.0,27930630,30008559,...,30263324,33260124,13480571,12209369,12979148,13578114,15854286,17271334,18331635,20688293
4,5,World,NaN,NaN,900,Eastern and South-Eastern Asia,1832,14465509.0,17262816,20822011,...,17926170,19677034,7395672,8889707,11060445,13077356,16213093,18793920,20297350,21732201


In [2]:
# Nome della colonna con il paese di origine
col_origine = "Region, development group, country or area of origin"

# Tengo solo le righe in cui l'origine è l'Albania
# .str.strip() toglie eventuali spazi prima e dopo il nome
albanesi = df[df[col_origine].astype(str).str.strip() == "Albania"]

print("Righe con origine Albania:", len(albanesi))

Righe con origine Albania: 81


In [3]:
# I paesi veri hanno un codice inferiore a 900;
# regioni e gruppi (World, Europe...) hanno codici da 900 in su
albanesi = albanesi[albanesi["Location code of destination"] < 900]

print("Paesi di destinazione:", len(albanesi))

Paesi di destinazione: 41


In [4]:
# Anni disponibili (totale maschi + femmine)
anni = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024]

# Tengo solo il paese di destinazione e gli anni
col_dest = "Region, development group, country or area of destination"
albanesi = albanesi[[col_dest] + anni]

# Rinomino la colonna del paese
albanesi = albanesi.rename(columns={col_dest: "paese"})

# Tolgo l'asterisco (*) che l'ONU usa per le note, es. "France*"
albanesi["paese"] = albanesi["paese"].str.replace("*", "", regex=False).str.strip()

# Ordino dal paese con più albanesi nel 2024
albanesi.sort_values(2024, ascending=False).head(10)

,paese,1990,1995,2000,2005,2010,2015,2020,2024
18805,Italy,60088.0,94077,141643,260914,386978,418672,493666,542638
18567,Greece,63963.0,230218,400253,423598,463157,436566,446614,474441
19074,North Macedonia,45001.0,58009,69367,69782,70230,67799,68113,68356
20202,France,1134.0,1840,3185,5816,7342,17618,34341,54967
25950,Canada,195.0,1178,4254,9657,13414,15850,17525,20718
20097,Belgium,691.0,865,1080,2946,6820,9614,12749,16175
17791,Sweden,150.0,243,351,647,1209,2320,7495,11986
19981,Austria,931.0,1285,1758,2368,2694,3508,4820,6148
26831,Australia,822.0,1119,1529,2281,2872,3447,3973,4105
19021,Montenegro,2511.0,2511,2511,2511,2015,2754,2763,2741


In [5]:
# Trasformo la tabella da formato "largo" a formato "lungo"
stock = albanesi.melt(
    id_vars="paese",          # colonna che resta uguale
    var_name="anno",          # le vecchie colonne diventano la colonna "anno"
    value_name="albanesi"     # i numeri vanno nella colonna "albanesi"
)

print("Righe e colonne:", stock.shape)
stock.head(10)

Righe e colonne: (328, 3)


,paese,anno,albanesi
0,Libya,1990,315.0
1,South Africa,1990,110.0
2,Guinea,1990,0.0
3,Mali,1990,0.0
4,Cyprus,1990,69.0
5,Georgia,1990,0.0
6,Bulgaria,1990,103.0
7,Hungary,1990,96.0
8,Romania,1990,633.0
9,Slovakia,1990,6.0


In [6]:
# Converto anno e albanesi in numeri interi (senza .0)
stock["anno"] = stock["anno"].astype(int)
stock["albanesi"] = stock["albanesi"].astype(int)

# Controllo i tipi delle colonne
print(stock.dtypes)

# Salvo la tabella pulita nella cartella dataclean
stock.to_csv("dataclean/stock_albanesi_1990_2024.csv", index=False)
print("File salvato!", len(stock), "righe")

paese       object
anno         int64
albanesi     int64
dtype: object
File salvato! 328 righe
